[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bhaskarjitsarmah/RL-Agents-Workshop-LLM/blob/main/notebooks/NB7_deployment.ipynb)

# NB7 - Deployment: Merge, Serve, Measure

An adapter in a Colab VM is not a system. This notebook turns it into one, and
then asks the question that decides whether any of this ships:

> **What does it cost per thousand queries, and how long does a user wait?**

Accuracy is one axis. Latency and cost are the other two, and the decision lives
on the Pareto front rather than at the top of a leaderboard.

> **Restart the runtime before this notebook.** Colab does not free GPU memory between notebooks, and a leftover model from the previous one is the most common cause of an out-of-memory error halfway through a training run.
>
> *Runtime -> Restart session*, then run the setup cell below.

In [ ]:
# --- Setup. Safe to re-run. ---------------------------------------------
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("/content/RL-Agents-Workshop-LLM"):
        subprocess.run(["git", "clone", "-q",
                        "https://github.com/bhaskarjitsarmah/RL-Agents-Workshop-LLM.git", "/content/RL-Agents-Workshop-LLM"], check=True)
    os.chdir("/content/RL-Agents-Workshop-LLM")
    # Colab's own keyring, read BEFORE preflight computes CAP -- otherwise the
    # notebook decides "no W&B key" while the key sits unread in the sidebar.
    # Absent secrets are normal, not an error: everything downgrades gracefully.
    try:
        from google.colab import userdata
        for _k in ("WANDB_API_KEY", "OPENAI_API_KEY", "HF_TOKEN"):
            try:
                os.environ.setdefault(_k, userdata.get(_k) or "")
            except Exception:
                pass
    except Exception:
        pass
    for _k in [k for k, v in list(os.environ.items()) if v == ""]:
        del os.environ[_k]          # empty != set; CAP tests truthiness

    # Install with uv, not pip: same resolution, several times faster on Colab.
    # The CORE layers on top of Colab's torch and never replaces it -- see the
    # header of requirements-colab.txt for why pinning torch broke this before.
    print("Installing the training stack with uv (1-2 min the first time)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    def _uv(*pkgs):
        return subprocess.run([sys.executable, "-m", "uv", "pip", "install",
                               "--system", "-q", *pkgs]).returncode

    # CORE -- must succeed. Fail loudly instead of continuing on stock packages:
    # a swallowed install failure surfaces 10 cells later as a dtype or
    # bitsandbytes error that names nothing resembling its cause.
    if _uv("-r", "requirements-colab.txt") != 0:
        print("*** CORE INSTALL FAILED -- scroll up for the uv error. ***")
        raise SystemExit("core install failed -- see WORKSHOP_GUIDE.md")

    # NB5's openpipe-art, best-effort: it can fail to resolve, and NB5 falls back
    # to a pre-baked run if it is absent. A failure here must not break the core.
    _uv("openpipe-art>=0.4.0")

    # Unsloth: ~2x faster LoRA on a T4, which is the difference between NB3
    # fitting in a lunch break and not. Installed HERE rather than in
    # requirements-colab.txt, and UNPINNED.
    #   * unpinned, because the old `unsloth==2024.12.4` pin required torch
    #     2.5.1 and was what made the entire install abort;
    #   * here rather than in the requirements file, because this is the one
    #     dependency heavy enough to fail on the day, and a failure has to
    #     degrade to the transformers + bitsandbytes path, not kill the core.
    # Skip it with:  os.environ["USE_UNSLOTH"] = "0"  above this cell.
    if os.environ.get("USE_UNSLOTH") != "0":
        if _uv("unsloth", "unsloth_zoo") != 0:
            print("unsloth did not install -- continuing on transformers + "
                  "bitsandbytes. Same adapter, slower. This is not an error.")

    # Unsloth CAN drag a different torch in. If it did, the kernel must restart
    # before anything imports torch, or you get a cryptic CUDA error later.
    from importlib.metadata import version as _ver
    if "torch" in sys.modules and sys.modules["torch"].__version__ != _ver("torch"):
        print("=" * 68)
        print("  torch was replaced. Runtime -> Restart session, then Run all again.")
        print("=" * 68)
        raise SystemExit("restart required -- see the message above")
else:
    # Run from the REPO ROOT in both environments, so every relative path in
    # every notebook ("data/...") means the same thing whether you are on Colab
    # (cwd = repo root) or opened the file locally from notebooks/.
    if os.path.basename(os.getcwd()) == "notebooks":
        os.chdir("..")
sys.path.insert(0, os.getcwd())

# Results that outlive the VM. Every Colab notebook is a SEPARATE runtime, so
# NB3 trains the GRPO curve into its own /content and NB5 -- a different VM --
# cannot see it. Anything one notebook computes for another has to land
# somewhere shared, and Drive is the only such place on free Colab.
# Set RESULTS_DIR before importing llm_utils: it is read at import time.
if IN_COLAB and os.environ.get("USE_DRIVE", "1") != "0":
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        _rd = "/content/drive/MyDrive/rl-workshop-results"
        os.makedirs(_rd, exist_ok=True)
        os.environ["RESULTS_DIR"] = _rd
        print(f"Results -> {_rd} (shared across notebooks, survives restarts)")
    except Exception as _e:
        print(f"Drive not mounted ({_e}). Results stay in this VM only, so a")
        print("later notebook will not see what this one computes. Not fatal.")

from llm_utils import (build_db, preflight, capability, load_result,
                       report_number, save_result)
from llm_utils.plotting import use_house_style
import matplotlib.pyplot as plt

CAP = preflight()
use_house_style()
print("Database ready at:", build_db())
if not CAP["gpu"]:
    print()
    print("No GPU detected -> REPLAY MODE.")
    print("Training cells will load pre-baked runs; every chart still renders.")
    print("In Colab: Runtime -> Change runtime type -> T4 GPU, then re-run.")

In [ ]:
def baked(key, how):
    """Load a pre-baked run, or explain exactly how to produce it.

    Returns None when the artifact is missing. Callers must check -- we would
    rather show no chart than an invented one.
    """
    data = load_result(key)
    if data is None:
        print(f"[{key}] not baked yet.")
        print(f"  Produce it with:  {how}")
        print("  Then re-run this cell. (The pre-baked files ship with the repo;")
        print("   you only need this if you are rebuilding them yourself.)")
    return data


PREBAKED = not CAP["gpu"]   # charts get a watermark when we are replaying

## 1. Merge the adapter

LoRA at `r=16` over seven projections on a 1.5B model is ~18M parameters - about
**36 MB in fp16**. That is why the pre-baked adapters download in seconds.

For serving we merge them into the base weights, which removes the adapter
indirection at inference time.

In [ ]:
from llm_utils.config import adapter_repo, base_model
print("base   :", base_model())
print("adapter:", adapter_repo("grpo"))

if CAP["gpu"]:
    from llm_utils.trainers import merge_and_save
    merged = merge_and_save(adapter_repo("grpo"), "out/merged-fp16")
else:
    merged = None
    print("\n(merge needs a GPU; the serving numbers below are pre-baked)")

### Verify the merge before trusting it

A merge bug is silent: the merged model still produces fluent SQL, just slightly
different SQL. It shows up here or it does not show up at all.

**The merged model must reproduce the adapter's test-16 accuracy exactly.**

In [ ]:
from llm_utils import evaluate
from llm_utils.local_llm import LocalLM, make_local_agent

if CAP["gpu"] and merged:
    lm_adapter = LocalLM(adapter=adapter_repo("grpo"))
    lm_merged = LocalLM(model_id=merged, load_in_4bit=False)
    a = evaluate(make_local_agent(lm_adapter), split="test")
    b = evaluate(make_local_agent(lm_merged), split="test")
    print(report_number(a, "adapter"))
    print(report_number(b, "merged "))
    same = [x["correct"] for x in sorted(a["records"], key=lambda r: r["id"])] == \
           [x["correct"] for x in sorted(b["records"], key=lambda r: r["id"])]
    print(f"\nper-item identical: {same}")
    if not same:
        print("!! MERGE BUG. Do not serve this. Check dtype and adapter path.")
else:
    v = baked("nb7_merge_check",
                  "python scripts/bake_all.py --stage deploy")
    if v:
        print("adapter vs merged, per-item identical:", v["identical"])

## 2. Serve it - three tiers, with fallback

| tier | when | note |
|---|---|---|
| **vLLM** | sm_80+ (A10, L4, A100) | fastest; unreliable on a Turing T4 |
| **FastAPI + transformers** | anywhere with a GPU | simple, adequate, what we usually get |
| **llama.cpp (GGUF)** | CPU only | slow but genuinely deployable on a laptop |

The notebook tries them in order and reports which one came up, rather than
assuming.

In [ ]:
served = baked("nb7_serving",
                  "python scripts/bake_all.py --stage deploy")
if served:
    print(f"tier that came up: {served['tier']}")
    print(f"reason: {served.get('reason', '-')}")

### Then score the served endpoint with the *vendored* `evaluate()`

Point `OPENAI_BASE_URL` at whatever came up, and run the same function that
produced repo 1's 0.75. If the served number differs from the in-process number,
something in the serving path is wrong - and you want to know that now, not from
a user.

In [ ]:
import os
from llm_utils import reset_client

if served and served.get("base_url") and CAP["gpu"]:
    os.environ["OPENAI_BASE_URL"] = served["base_url"]
    reset_client()                       # or we keep talking to the old endpoint
    lm_served = LocalLM(backend="openai", base_url=served["base_url"],
                        api_model=served.get("model"))
    res_served = evaluate(make_local_agent(lm_served), split="test")
    print(report_number(res_served, "served endpoint"))
else:
    v = baked("nb7_served_eval",
                  "python scripts/bake_all.py --stage deploy")
    if v:
        print(report_number(tuple(v["test16"]), "served endpoint"))
        print("in-process vs served, identical:", v.get("matches_in_process"))

## 3. Latency and throughput

In [ ]:
lat = baked("nb7_latency",
                  "python scripts/bake_all.py --stage deploy")
if lat:
    fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
    for name, xs in lat["latency_ms"].items():
        ax[0].hist(xs, bins=24, alpha=0.6, label=name)
    ax[0].set_xlabel("latency (ms)"); ax[0].set_title("per-query latency")
    ax[0].legend()
    bs = list(lat["throughput"])
    ax[1].plot(bs, [lat["throughput"][b] for b in bs], marker="o", color="#DD8452")
    ax[1].set_xlabel("batch size"); ax[1].set_ylabel("queries / second")
    ax[1].set_title("throughput vs batch size")
    plt.tight_layout(); plt.show()

    for name, xs in lat["latency_ms"].items():
        s = sorted(xs)
        print(f"  {name:<22} p50 {s[len(s)//2]:>7.0f} ms   "
              f"p95 {s[int(len(s)*0.95)]:>7.0f} ms")

## 4. The number your manager will ask for

Self-hosting has a fixed hourly cost whether or not anyone is using it. An API
has a marginal cost per token and none when idle.

So self-hosting wins **above a break-even QPS** and loses below it. Compute it
rather than asserting it.

In [ ]:
from llm_utils.llm import GPU_HOURLY_USD, PRICING_PER_1M

if lat:
    qps = max(lat["throughput"].values())
    gpu_hr = GPU_HOURLY_USD["T4"]
    self_cost_1k = gpu_hr / (qps * 3600) * 1000

    in_tok, out_tok = lat.get("mean_prompt_tokens", 700), lat.get("mean_completion_tokens", 60)
    p = PRICING_PER_1M["gpt-4o-mini"]
    api_cost_1k = (in_tok * p["in"] + out_tok * p["out"]) / 1e6 * 1000

    print(f"self-hosted T4  : {qps:.2f} q/s at ${gpu_hr}/hr -> ${self_cost_1k:.3f} / 1k queries")
    print(f"gpt-4o-mini API : {in_tok} in + {out_tok} out tokens -> ${api_cost_1k:.3f} / 1k queries")
    breakeven = gpu_hr / (api_cost_1k / 1000 * 3600)
    print(f"\nbreak-even: ~{breakeven:.2f} queries/second sustained.")
    print("Below that, the API is cheaper. Above it, the GPU is.")
    print("This is the slide that decides the project, and it is one division.")

In [ ]:
pareto_pts = baked("nb7_pareto",
                  "python scripts/bake_all.py --stage deploy")
if pareto_pts:
    from llm_utils.plotting import pareto
    pareto(pareto_pts, title="Accuracy vs cost per 1k queries", prebaked=PREBAKED)
    plt.show()

### What the Pareto front actually says

A fine-tuned 1.5B that matches a much larger API model is not interesting because
it is *better*. It is interesting because it is **the smallest thing that holds
that accuracy**, and small is what makes it cheap, fast, on-prem-able, and yours.

That, and not a leaderboard position, is the case for optimizing the weights.

## Takeaways

1. **Verify the merge per-item.** A merge bug is silent - the model still writes fluent SQL, just subtly different SQL.
2. Score the **served** endpoint with the vendored `evaluate()`. A serving path that changes the number is a bug you want to find before a user does.
3. Serving has tiers and the fast one is not always available. Try, measure, and report which came up.
4. **Compute the break-even QPS.** Self-hosting is a fixed cost and an API is marginal; that one division decides the architecture.
5. The value of a fine-tuned small model is not that it wins, it is that it is **the smallest thing that holds the accuracy**.

### The gap this leaves (-> NB8)

Two workshops. Two philosophies. One scoreboard that has been identical the
whole way through.

Time to settle it - and to find out whether the question "harness or weights?"
was even the right question.

### Exercise

1. Re-run the break-even calculation for an A10 at $1.00/hr. Does the answer
   change the architecture you would choose?
2. Quantise the merged model to 4-bit for serving and re-score. How much accuracy
   does the compression cost, and how much latency does it buy?
3. Your traffic is 0.2 QPS at midday and 0.001 QPS overnight. What do you deploy?
   (There is more than one defensible answer.)

In [ ]:
# --- Cost / throughput meter -------------------------------------------
from llm_utils import METER, flush
print(METER)          # OpenAI spend (0 unless you ran the comparison rows)
flush()